<a href="https://colab.research.google.com/github/HoaiAn001/CV_TDTU_Final/blob/main/CV_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Install Required Libraries
Install the necessary packages for object detection, tracking, and evaluation:
- **ultralytics**: YOLO model and BoT-SORT tracker
- **lapx**: Linear Assignment Problem solver (required by BoT-SORT)
- **TrackEval**: Official MOT evaluation toolkit for HOTA, MOTA, IDF1 metrics


In [ ]:
!pip install ultralytics lap -q

## 2. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Define Paths and Directory Structure

In [ ]:
import os

BASE         = '/content/drive/MyDrive/CV_Final'
IMG_DIR      = f'{BASE}/data/img1'
GT_DIR       = f'{BASE}/data/gt/gt.txt'

OUTPUT_VIDEO = f'{BASE}/results/output_tracking.mp4'
TXT_OUTPUT   = f'{BASE}/results/MOT17-02-FRCNN.txt'
CHECKPOINT_PATH = f'{BASE}/results/checkpoint.json'

os.makedirs(f'{BASE}/results', exist_ok=True)

print(f"IMG_DIR : {IMG_DIR}")
print(f"GT_DIR  : {GT_DIR}")
print(f"Exists IMG_DIR : {os.path.exists(IMG_DIR)}")
print(f"Exists GT_DIR  : {os.path.exists(GT_DIR)}")

In [ ]:
import yaml, os

custom_botsort = {
    'tracker_type': 'botsort',

    # Ngưỡng detection
    'track_high_thresh': 0.2,
    'track_low_thresh' : 0.05,
    'new_track_thresh' : 0.4,

    # Buffer
    'track_buffer'     : 90,

    # Matching
    'match_thresh'     : 0.7,
    'fuse_score'       : True,

    # BoT-SORT specific
    'gmc_method'       : 'sparseOptFlow',
    'proximity_thresh' : 0.5,
    'appearance_thresh': 0.25,
    'with_reid'        : False,
}

CUSTOM_YAML = f'{BASE}/botsort_custom.yaml'
with open(CUSTOM_YAML, 'w') as f:
    yaml.dump(custom_botsort, f, default_flow_style=False)

print("botsort_custom.yaml was created!")
print(yaml.dump(custom_botsort))

## 4. Multi-Object Tracking with YOLO26 + BoT-SORT

Run multi-object tracking on the MOT17-02-FRCNN dataset using:
- **Model**: YOLO26m — the latest Ultralytics detection model, optimized for edge deployment
- **Tracker**: BoT-SORT (`botsort.yaml`) — robust tracker combining Kalman Filter and ReID
- **Parameters**: `conf=0.3`, `iou=0.5` as required, `classes=0` to track pedestrians only

For each frame, the tracker outputs bounding boxes and track IDs, which are:
1. Written to a `.txt` file in MOTChallenge format: `frame, id, x, y, w, h, conf, -1, -1, -1`
2. Drawn on the frame and saved to the output video

A checkpoint mechanism is included to resume tracking automatically if the Colab session is disconnected.

In [ ]:
from ultralytics import YOLO
from tqdm import tqdm
import cv2, os, json

model = YOLO('yolo26x-seq.pt')

frames = sorted([
    os.path.join(IMG_DIR, f)
    for f in os.listdir(IMG_DIR)
    if f.lower().endswith('.jpg')
])
print(f"Total number of frames: {len(frames)}")

if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH, 'r') as f:
        ckpt = json.load(f)
    start_frame = ckpt['last_frame']
    print(f"Resuming from frame {start_frame + 1}/{len(frames)}")
else:
    start_frame = 0
    print("Starting from the beginning.")

first_img = cv2.imread(frames[0])
h, w = first_img.shape[:2]
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
video_writer = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, 30, (w, h))

txt_mode = 'a' if start_frame > 0 else 'w'
txt_file = open(TXT_OUTPUT, txt_mode)

SAVE_EVERY = 50

pbar = tqdm(
    enumerate(frames, start=1),
    total=len(frames),
    initial=start_frame,
    desc='Tracking',
    unit='frame'
)

for frame_id, frame_path in pbar:
    if frame_id <= start_frame:
        continue

    results = model.track(
        source=frame_path,
        tracker=CUSTOM_YAML,
        conf=0.3,
        iou=0.5,
        classes=0,
        persist=True,
        imgsz=1920,
        verbose=False
    )

    boxes = results[0].boxes
    n_tracks = 0

    if boxes is not None and boxes.id is not None:
        n_tracks = len(boxes.id)
        for box, track_id, conf in zip(
            boxes.xyxy.cpu().numpy(),
            boxes.id.cpu().numpy().astype(int),
            boxes.conf.cpu().numpy()
        ):
            x1, y1, x2, y2 = box
            bw, bh = x2 - x1, y2 - y1
            txt_file.write(
                f"{frame_id},{track_id},{x1:.2f},{y1:.2f},"
                f"{bw:.2f},{bh:.2f},{conf:.4f},-1,-1,-1\n"
            )

    annotated = results[0].plot()
    video_writer.write(annotated)
    pbar.set_postfix({'tracks': n_tracks, 'frame': frame_id})

    if frame_id % SAVE_EVERY == 0:
        txt_file.flush()
        with open(CHECKPOINT_PATH, 'w') as f:
            json.dump({'last_frame': frame_id, 'total': len(frames)}, f)

txt_file.close()
video_writer.release()
pbar.close()

if os.path.exists(CHECKPOINT_PATH):
    os.remove(CHECKPOINT_PATH)

print(f"\nVideo : {OUTPUT_VIDEO}")
print(f"Text : {TXT_OUTPUT}")

## 5. Prepare Directory Structure for TrackEval

In [ ]:
!pip uninstall trackeval -y -q
!pip install git+https://github.com/JonathonLuiten/TrackEval.git -q

In [ ]:
import shutil

TRACKEVAL_BASE = f'{BASE}/trackeval'

TRACKER_DIR = f'{TRACKEVAL_BASE}/trackers/MOT17-train/botsort/data'
GT_OUT_DIR  = f'{TRACKEVAL_BASE}/gt/MOT17-train/MOT17-02-FRCNN/gt'
SEQMAP_DIR = f'{TRACKEVAL_BASE}/gt/seqmaps'

os.makedirs(TRACKER_DIR, exist_ok=True)
os.makedirs(GT_OUT_DIR,  exist_ok=True)
os.makedirs(SEQMAP_DIR, exist_ok=True)

shutil.copy(TXT_OUTPUT, f'{TRACKER_DIR}/MOT17-02-FRCNN.txt')
shutil.copy(GT_DIR, f'{GT_OUT_DIR}/gt.txt')

frames_count = len(os.listdir(IMG_DIR))
seqinfo_content = f"""[Sequence]
name=MOT17-02-FRCNN
imDir=img1
frameRate=30
seqLength={frames_count}
imWidth=1920
imHeight=1080
imExt=.jpg
"""
seqinfo_path = f'{TRACKEVAL_BASE}/gt/MOT17-train/MOT17-02-FRCNN/seqinfo.ini'
with open(seqinfo_path, 'w') as f:
    f.write(seqinfo_content)

with open(f'{SEQMAP_DIR}/MOT17-train.txt', 'w') as f:
    f.write("name\n")
    f.write("MOT17-02-FRCNN\n")

print(f"seqLength = {frames_count} frames")
print("Folder created successfully")

## 6. Evaluate Tracking Performance (HOTA, MOTA, IDF1)

Evaluate the tracker's output against the ground truth using three standard MOT metrics:

| Metric | Full Name | Measures |
|--------|-----------|----------|
| **HOTA** | Higher Order Tracking Accuracy | Overall performance (detection + association) |
| **MOTA** | Multiple Object Tracking Accuracy | Detection accuracy (FP, FN, ID switches) |
| **IDF1** | Identity F1 Score | Association accuracy (how well IDs are maintained) |

The evaluation is performed using the [TrackEval](https://github.com/JonathonLuiten/TrackEval) library,
which is the official evaluation toolkit used in the MOTChallenge benchmark.

In [ ]:
import numpy as np
# Monkey-patch các alias bị deprecated
np.float = np.float64
np.int   = np.int64
np.bool  = bool
np.complex = np.complex128
np.object  = object
np.str     = str

import trackeval
import trackeval.metrics as metrics
from trackeval import Evaluator, datasets, metrics as tm

# Cấu hình dataset
dataset_config = trackeval.datasets.MotChallenge2DBox.get_default_dataset_config()
dataset_config.update({
    'GT_FOLDER'      : f'{TRACKEVAL_BASE}/gt',
    'TRACKERS_FOLDER': f'{TRACKEVAL_BASE}/trackers',
    'BENCHMARK'      : 'MOT17',
    'SPLIT_TO_EVAL'  : 'train',
    'TRACKERS_TO_EVAL': ['botsort'],
    'CLASSES_TO_EVAL': ['pedestrian'],
    'PRINT_CONFIG'   : False,
})

# Cấu hình evaluator
eval_config = trackeval.Evaluator.get_default_eval_config()
eval_config.update({
    'DISPLAY_LESS_PROGRESS': False,
    'PRINT_CONFIG'         : False,
})

# Các metrics cần tính
metrics_list = [
    trackeval.metrics.HOTA(),   # HOTA: overall performance
    trackeval.metrics.CLEAR(),  # MOTA nằm trong CLEAR
    trackeval.metrics.Identity(), # IDF1 nằm trong Identity
]

# Chạy đánh giá
evaluator  = trackeval.Evaluator(eval_config)
dataset    = [trackeval.datasets.MotChallenge2DBox(dataset_config)]
results, _ = evaluator.evaluate(dataset, metrics_list)

# In kết quả quan trọng
res = results['MotChallenge2DBox']['botsort']['COMBINED_SEQ']['pedestrian']

print("\n" + "="*45)
print("KẾT QUẢ ĐÁNH GIÁ MOT METRICS")
print("="*45)
print(f"HOTA  : {res['HOTA']['HOTA'].mean()*100:.2f}%   (overall performance)")
print(f"MOTA  : {res['CLEAR']['MOTA']*100:.2f}%   (detection accuracy)")
print(f"IDF1  : {res['Identity']['IDF1']*100:.2f}%   (association accuracy)")
print("="*45)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Dữ liệu của 3 lần thực nghiệm
labels = ['Baseline\n(YOLO26n + 640px)', 'Exp 1\n(YOLO26m + 640px)', 'Exp 2\n(YOLO26x + 1920px)']
hota = [26.67, 33.68, 34.27]
mota = [18.67, 26.87, 34.77]
idf1 = [26.67, 38.69, 43.85]

x = np.arange(len(labels))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 6))

# Vẽ các cột
rects1 = ax.bar(x - width, hota, width, label='HOTA (%)', color='#4C72B0')
rects2 = ax.bar(x, mota, width, label='MOTA (%)', color='#55A868')
rects3 = ax.bar(x + width, idf1, width, label='IDF1 (%)', color='#C44E52')

# Trang trí biểu đồ
ax.set_ylabel('Scores (%)', fontsize=12, fontweight='bold')
ax.set_title('Ablation Study: Tracking Performance across Model Configurations', fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim(0, 55)
ax.grid(axis='y', linestyle='--', alpha=0.7)

# Hiển thị số liệu trên đỉnh cột
def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height}%',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),  # 3 points vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=10)

autolabel(rects1)
autolabel(rects2)
autolabel(rects3)

fig.tight_layout()
plt.show()

## 7. Display Output Video

Load and display the tracking output video directly in the notebook.
The video shows bounding boxes and track IDs overlaid on each frame,
allowing visual inspection of the tracker's performance.

In [ ]:
import os
from IPython.display import HTML
from base64 import b64encode

input_video = f'{BASE}/results/output_tracking.mp4'
web_video = f'{BASE}/results/output_web.mp4'

!ffmpeg -y -i "{input_video}" -vcodec libx264 "{web_video}" -loglevel quiet

if os.path.exists(web_video):
    mp4 = open(web_video, "rb").read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

    display(HTML(f"""
    <div style="text-align: center;">
        <video width=800 controls>
            <source src="{data_url}" type="video/mp4">
            Trình duyệt của bạn không hỗ trợ thẻ video.
        </video>
    </div>
    """))
else:
    print(f"Lỗi: Không tìm thấy file tại {web_video}")